# 감성분석(Sentiment Analysis)

- 텍스트(리뷰, 댓글, 기사 등)에 담긴 **긍정/부정/중립** 같은 전반적 태도(Sentiment)를 분류하는 분석
- 핵심은 **텍스트 전체의 긍·부정 경향을 파악하는 것**

- [참고] 감정 분석 (Emotion Analysis)
    - 텍스트 안에서 구체적인 **감정(emotion)** 을 분류하는 분석. (예: 행복, 분노, 슬픔, 두려움, 놀람, 혐오 등)
    - 단순 긍·부정이 아니라 세분화된 감정 상태를 구분하는 것

- 감성분석은 난이도에 따라 단순한 긍/부정에서 유형별 분류까지 나아감
- 복잡한 분류일 경우 머신러닝/딥러닝의 알고리즘 또는 정교한 어휘사전이 필요함


# 1.라이브러리 불러오기

In [1]:
import pandas as pd
import re
from konlpy.tag import Okt

# 2.데이터 셋 불러오기

- 네이버 영화 평점 데이터 셋
    - https://github.com/e9t/nsmc  

In [2]:
train_df = pd.read_csv('result_sentiment.csv')
train_df.head()


,작성자,평점,날짜,리뷰내용,정제리뷰,긍정확률,예측감성,평점감성
0,강성원,1,2026-05-30 23:38:18,오늘의 포인트를 하려면 업데이트를.하라고 문구가 뜨는데 업데이트가 되지 않음,오늘의 포인트를 하려면 업데이트를하라고 문구가 뜨는데 업데이트가 되지 않음,0.173091,부정,부정
1,고태성,4,2026-05-30 23:02:01,좋아요,좋아요,0.993156,긍정,긍정
2,김기형,5,2026-05-30 22:56:35,편리함,편리함,0.445792,부정,긍정
3,최경자,5,2026-05-30 22:49:31,재미있어요,재미있어요,0.997579,긍정,긍정
4,정현정카슈,3,2026-05-30 22:44:51,잘모르겠음,잘모르겠음,0.322276,부정,중립


In [3]:
# '평점 감성' 컬럼의 긍정과 부정을 숫자로 변환
train_df['label'] = train_df['평점감성'].map({
    '긍정': 1,
    '부정': 0
})
train_df.dropna(subset=['label'], inplace=True)
train_df.head()

,작성자,평점,날짜,리뷰내용,정제리뷰,긍정확률,예측감성,평점감성,label
0,강성원,1,2026-05-30 23:38:18,오늘의 포인트를 하려면 업데이트를.하라고 문구가 뜨는데 업데이트가 되지 않음,오늘의 포인트를 하려면 업데이트를하라고 문구가 뜨는데 업데이트가 되지 않음,0.173091,부정,부정,0.0
1,고태성,4,2026-05-30 23:02:01,좋아요,좋아요,0.993156,긍정,긍정,1.0
2,김기형,5,2026-05-30 22:56:35,편리함,편리함,0.445792,부정,긍정,1.0
3,최경자,5,2026-05-30 22:49:31,재미있어요,재미있어요,0.997579,긍정,긍정,1.0
5,Insuk Kwon,5,2026-05-30 22:39:41,수수료없고 매일 이자주고 다 좋은데 방송봤는데 포인트 안들어오구ㅡㅡ업뎃하래서 했는데...,수수료없고 매일 이자주고 다 좋은데 방송봤는데 포인트 안들어오구ㅡㅡ업뎃하래서 했는데...,0.048068,부정,긍정,1.0


In [4]:
train_df.shape

(133, 9)

In [5]:
train_df['label'].value_counts()

label
1.0    92
0.0    41
Name: count, dtype: int64

In [6]:
train_df.info()

<class 'pandas.DataFrame'>
Index: 133 entries, 0 to 144
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   작성자     133 non-null    str    
 1   평점      133 non-null    int64  
 2   날짜      133 non-null    str    
 3   리뷰내용    133 non-null    str    
 4   정제리뷰    130 non-null    str    
 5   긍정확률    133 non-null    float64
 6   예측감성    133 non-null    str    
 7   평점감성    133 non-null    str    
 8   label   133 non-null    float64
dtypes: float64(2), int64(1), str(6)
memory usage: 10.4 KB


# 3.데이터 전처리

In [7]:
# 리뷰 결측치를 빈칸으로 처리
train_df = train_df.fillna(' ')

In [8]:
train_df.isnull().sum()

작성자      0
평점       0
날짜       0
리뷰내용     0
정제리뷰     0
긍정확률     0
예측감성     0
평점감성     0
label    0
dtype: int64

In [9]:
result = []
for temp in train_df['리뷰내용']:
    kor_str = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', temp)
    result.append(kor_str)

In [10]:
train_df['리뷰내용'] = train_df['리뷰내용'].apply(lambda x : re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', x) )
train_df

,작성자,평점,날짜,리뷰내용,정제리뷰,긍정확률,예측감성,평점감성,label
0,강성원,1,2026-05-30 23:38:18,오늘의 포인트를 하려면 업데이트를하라고 문구가 뜨는데 업데이트가 되지 않음,오늘의 포인트를 하려면 업데이트를하라고 문구가 뜨는데 업데이트가 되지 않음,0.173091,부정,부정,0.0
1,고태성,4,2026-05-30 23:02:01,좋아요,좋아요,0.993156,긍정,긍정,1.0
2,김기형,5,2026-05-30 22:56:35,편리함,편리함,0.445792,부정,긍정,1.0
3,최경자,5,2026-05-30 22:49:31,재미있어요,재미있어요,0.997579,긍정,긍정,1.0
5,Insuk Kwon,5,2026-05-30 22:39:41,수수료없고 매일 이자주고 다 좋은데 방송봤는데 포인트 안들어오구ㅡㅡ업뎃하래서 했는데...,수수료없고 매일 이자주고 다 좋은데 방송봤는데 포인트 안들어오구ㅡㅡ업뎃하래서 했는데...,0.048068,부정,긍정,1.0
...,...,...,...,...,...,...,...,...,...
140,jiheon Song,5,2026-05-30 00:15:00,앱은 편리하고 좋은데 관리하시는분들이 너무 편하게 일하려는듯한 느낌을받았어요이용자가...,앱은 편리하고 좋은데 관리하시는분들이 너무 편하게 일하려는듯한 느낌을받았어요이용자가...,0.822041,긍정,긍정,1.0
141,이동훈,5,2026-05-30 00:13:50,,,0.445792,부정,긍정,1.0
142,문재순,1,2026-05-30 00:12:14,했는데도 계속 업데이트 하라고 뜨네요ㅠ,했는데도 계속 업데이트 하라고 뜨네요ㅠ,0.643941,긍정,부정,0.0
143,조은새,5,2026-05-30 00:10:41,앱은 작동하고 서비스가있지만다음내용화면이 알수없는것,앱은 작동하고 서비스가있지만다음내용화면이 알수없는것,0.432421,부정,긍정,1.0


In [11]:
test_df = pd.read_csv('result_sentiment.csv')
test_df['label'] = test_df['평점감성'].map({
    '긍정': 1,
    '부정': 0
})
test_df = test_df.fillna(' ')
test_df['리뷰내용'] = test_df['리뷰내용'].apply(lambda x : re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', x))

In [12]:
test_df.head()

,작성자,평점,날짜,리뷰내용,정제리뷰,긍정확률,예측감성,평점감성,label
0,강성원,1,2026-05-30 23:38:18,오늘의 포인트를 하려면 업데이트를하라고 문구가 뜨는데 업데이트가 되지 않음,오늘의 포인트를 하려면 업데이트를하라고 문구가 뜨는데 업데이트가 되지 않음,0.173091,부정,부정,0.0
1,고태성,4,2026-05-30 23:02:01,좋아요,좋아요,0.993156,긍정,긍정,1.0
2,김기형,5,2026-05-30 22:56:35,편리함,편리함,0.445792,부정,긍정,1.0
3,최경자,5,2026-05-30 22:49:31,재미있어요,재미있어요,0.997579,긍정,긍정,1.0
4,정현정카슈,3,2026-05-30 22:44:51,잘모르겠음,잘모르겠음,0.322276,부정,중립,


In [13]:
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   작성자     145 non-null    str    
 1   평점      145 non-null    int64  
 2   날짜      145 non-null    str    
 3   리뷰내용    145 non-null    str    
 4   정제리뷰    145 non-null    str    
 5   긍정확률    145 non-null    float64
 6   예측감성    145 non-null    str    
 7   평점감성    145 non-null    str    
 8   label   145 non-null    object 
dtypes: float64(1), int64(1), object(1), str(6)
memory usage: 10.3+ KB


In [14]:
drop_cols = ['작성자', '평점', '날짜']

train_df.drop(columns=drop_cols, errors='ignore', inplace=True)
test_df.drop(columns=drop_cols, errors='ignore', inplace=True)

In [15]:
# 형태소 분석
okt = Okt()

def okt_tokenizer(text):
    token_kor = okt.morphs(text)
    return token_kor

# 4.TF-IDF
## 4.1. TF-IDF란?
- TF-IDF는 문서에서 **어떤 단어가 중요한 단어인지 수치로 표현하는 방법** 
- 텍스트를 머신러닝 모델에 넣기 위해 단어를 숫자로 바꿀 때 자주 사용
- 단순히 많이 나온 단어를 중요하다고 보는 것이 아니라, **특정 문서에서 자주 나오면서 전체 문서에서는 너무 흔하지 않은 단어** 를 중요하게 본다.
- `TfidfVectorizer`로 텍스트를 수치형 데이터로 변환

## 4.2. 왜 TF-IDF가 필요한가?

- 단어 빈도만 사용하면 문제가 생긴다.
- 예를 들어 리뷰 데이터에서 `영화`, `정말`, `너무`, `그리고` 같은 단어는 자주 등장할 수 있다.
- 하지만 이런 단어들은 대부분의 문서에 널리 등장하므로 특정 문서의 특징을 잘 설명하지 못한다.
- 반대로 `감동`, `지루`, `연기`, `스토리`, `최악`, `명작` 같은 단어는 특정 리뷰의 성격을 더 잘 설명할 수 있다.
- TF-IDF는 이런 단어에 더 높은 점수를 부여한다.
---

## 4.3. [참고]TF-IDF의 핵심 아이디어

- TF-IDF는 두 값을 곱해서 만든다.

$$
TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)
$$

- `t`: 단어(term)
- `d`: 문서(document)
- `TF(t, d)`: 특정 문서 안에서 단어가 얼마나 자주 등장하는지
- `IDF(t)`: 전체 문서에서 그 단어가 얼마나 희귀한지



#### 4.3.1. TF: Term Frequency

- TF는 **특정 문서 안에서 단어가 등장한 빈도**이다.
- 어떤 단어가 한 문서 안에서 많이 등장할수록 TF 값이 커진다.

가장 단순한 TF 계산식:

$$
TF(t, d) = \text{문서 } d \text{ 안에서 단어 } t \text{가 등장한 횟수}
$$

문서 길이를 고려한 TF 계산식:

$$
TF(t, d) =
\frac{
\text{문서 } d \text{ 안에서 단어 } t \text{가 등장한 횟수}
}{
\text{문서 } d \text{의 전체 단어 수}
}
$$

예:

- 문서 A: `이 영화 영화 정말 재미있다`
- 문서 A의 전체 단어 수: 5개
- `영화` 등장 횟수: 2번

$$
TF(\text{영화}, A) = \frac{2}{5} = 0.4
$$


#### 4.3.2. IDF: Inverse Document Frequency

- IDF는 **전체 문서 중에서 해당 단어가 얼마나 드문 단어인지**를 나타낸다.
- 많은 문서에 등장하는 흔한 단어는 IDF 값이 낮다.
- 적은 문서에만 등장하는 단어는 IDF 값이 높다.

기본 IDF 계산식:

$$
IDF(t) = \log \left( \frac{N}{DF(t)} \right)
$$

- `N`: 전체 문서 개수
- `DF(t)`: 단어 `t`가 등장한 문서 개수
- `log`: 값이 너무 커지는 것을 줄이기 위한 로그 함수

예:

- 전체 문서 수 `N = 1000`
- `영화`가 등장한 문서 수 `DF(영화) = 900`
- `감동`이 등장한 문서 수 `DF(감동) = 100`

$$
IDF(\text{영화}) =
\log \left( \frac{1000}{900} \right)
$$

$$
IDF(\text{감동}) =
\log \left( \frac{1000}{100} \right)
$$

- `영화`는 대부분의 문서에 등장하므로 IDF가 낮다.
- `감동`은 상대적으로 적은 문서에 등장하므로 IDF가 높다.

---


In [16]:
# TfidVectorizer는 텍스트 문장을 TF-IDF 숫자 벡터로 변환하는 도구
from sklearn.feature_extraction.text import TfidfVectorizer

In [17]:
# 리뷰 문장을 숫자 벡터로 변환
# 머신러닝 모델은 텍스트를 직접 학습할 수 없으므로 숫자로 바꿔야 됨.
tfidf_vect = TfidfVectorizer(tokenizer=okt_tokenizer,       # Okt 기반의 토큰화 함수 사용
                             ngram_range=(1,2),             # 단어 1개와 단어 2개를 모두 포함한 특징을 생성
                             max_df=0.9,                    # 전체 문서에서 90% 이상 나타나는 단어는 제외. 불필요한 흔한 단어는 제거
                             min_df=3)                      # 최소 3번 이상 등장하는 단어만 사용

In [18]:
# fit : 학습 데이터에서 단어 사전과 IDF 값을 학습
# 이 단계에서는 아직 문장을 행렬로 바꾸는 것이 아니라, 어떤 단어를 특징으로 쓸지 기준으로 잡는다.
tfidf_vect.fit(train_df['리뷰내용'])

c:\Users\playdata2\work_space(playdata)\SKN30_playdata\LLM\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",<function okt...0016F46B91440>
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyz

In [19]:
# transform : 학습 리뷰를 TF-IDF 숫자 벡터로 변환
# 결과는 문서 수 * 단어 특징 수 형태의 희소 행렬로 저장.
tfidf_vect_train = tfidf_vect.transform(train_df['리뷰내용'])

# 4.Logistic Regression

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

In [21]:
lr = LogisticRegression(random_state=1,
                        solver='liblinear')    # 이진분류 작업

In [22]:
# GridSearchCV를 사용해 최적의 'C'값 찾기
# C값이 작을 수록 규제가 강해지고, 값이 클수록 약해짐
params = {'C': [1, 2, 3.5, 4.5, 5.5, 9, 10]}

grid_cv = GridSearchCV(lr,                      # 로지스틱 회귀 모델
                       param_grid=params,       # C값 후보
                       cv=5,                    # 5-Fold 교차 검증 수행
                       scoring='accuracy',      # 평가지표로 정확도 사용
                       verbose=1)               # 진행상황 출력

In [23]:
# Tf-IDF로 변환된 리뷰 벡터와 정답 라벨을 이용해 학습
grid_cv.fit(tfidf_vect_train, train_df['label'])

Fitting 5 folds for each of 7 candidates, totalling 35 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...r='liblinear')
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [1, 2, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate paramete

In [24]:
print(grid_cv.best_params_, round(grid_cv.best_score_, 5))

{'C': 1} 0.82678


In [25]:
tfidf_matrix_test = tfidf_vect.transform(test_df['리뷰내용'])

In [26]:
best_estimator = grid_cv.best_estimator_

In [27]:
# 테스트 데이터에 대한 긍/부정 예측 수행
pred = best_estimator.predict(tfidf_matrix_test)

In [29]:
import numpy as np

print(test_df['label'].dtype, test_df['label'].shape)
print(type(pred), np.asarray(pred).dtype, np.asarray(pred).shape)
print(test_df['label'].head().tolist())
print(np.asarray(pred)[:5])

object (145,)
<class 'numpy.ndarray'> float64 (145,)
[0.0, 1.0, 1.0, 1.0, ' ']
[0. 1. 1. 1. 1.]


In [30]:
# 사용자 입력 리뷰에 대한 감성 분석 함수
def pred_sentiment(review):
    # 1. 데이터 클리닝
    review = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', review)

    # 2. Tf-idf 변환
    tfidf_review = tfidf_vect.transform([review])
    
    # 3. 감성 예측
    prediction = best_estimator.predict(tfidf_review)[0]
    print(best_estimator.predict_proba(tfidf_review))

    # 4. 결과 반환(1:긍정, 0: 부정)
    if prediction == 1:
        return '긍정 리뷰입니다.'
    else:
        return '부정 리뷰입니다.'



In [33]:
user_review = '이런 걸 돈주고 만드냐? 진짜 쓰레기 앱이다.'
user_review2 = '이거 뭐하는 앱이고 베네핏도 계속 줄여가면서 시간만 낭비하는 앱 ㅉㅉ'

In [37]:
pred_sentiment(user_review)

[[0.18336154 0.81663846]]


'긍정 리뷰입니다.'

# 데이터가 잘못(크롤링 결과 데이터 row수) 되었거나 코드 cell 중간 긍정, 부정 학습이 잘못 된 것으로 보인다.